# 03 — Modelo Baseline para Clusterização

Este notebook apresenta um template didático para criação de um modelo baseline de **clusterização**.

Use este notebook quando o objetivo for encontrar **grupos naturais nos dados**, sem uma variável-alvo conhecida.

Exemplos:

- segmentar clientes por comportamento;
- agrupar produtos por padrão de venda;
- agrupar alunos por desempenho;
- identificar perfis semelhantes.

## Premissa

Este template pressupõe que o dataset já passou pela etapa de preparação dos dados.

Ou seja:

- não há valores nulos relevantes;
- as variáveis categóricas já foram tratadas;
- as variáveis estão prontas para o modelo;
- não existe variável-alvo a ser prevista.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## 1. Carregar o dataset

Ajuste o caminho do arquivo conforme o local onde o dataset estiver salvo.

Exemplos de datasets didáticos para clusterização:

- Mall Customer Segmentation Data;
- Mall Customers Dataset;
- Customer Clustering.

In [ ]:
df = pd.read_csv("dataset_clusterizacao.csv")

df.head()

## 2. Visualização inicial dos dados

In [ ]:
print("Linhas e colunas:", df.shape)

df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.isnull().sum()

## 3. Selecionar variáveis para clusterização

Como não existe variável-alvo, usamos as variáveis que representam o perfil dos registros.

Remova colunas identificadoras, como `id_cliente`, `customer_id`, `nome` etc.

In [ ]:
colunas_remover = []  # Exemplo: ["id_cliente", "nome"]

X_cluster = df.drop(columns=colunas_remover)

X_cluster.head()

## 4. Visualização gráfica dos dados

In [ ]:
X_cluster.select_dtypes(include=np.number).hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

In [ ]:
X_cluster.select_dtypes(include=np.number).plot(
    kind="box",
    figsize=(12, 6),
    rot=45
)

plt.title("Boxplot das variáveis numéricas")
plt.show()

## 5. Visualização 2D simples

Se o dataset tiver pelo menos duas variáveis, podemos visualizar as duas primeiras.

In [ ]:
if X_cluster.shape[1] >= 2:
    plt.scatter(
        X_cluster.iloc[:, 0],
        X_cluster.iloc[:, 1]
    )

    plt.xlabel(X_cluster.columns[0])
    plt.ylabel(X_cluster.columns[1])
    plt.title("Visualização inicial dos dados")
    plt.show()
else:
    print("O dataset possui menos de duas variáveis para visualização 2D.")

## 6. Normalização

Para KMeans, a normalização é muito importante, porque o algoritmo usa distância entre pontos.

Sem normalização, variáveis com escala maior podem dominar o agrupamento.

Regra:

- use `fit_transform` no conjunto completo usado para clusterização;
- se houver novos registros depois, use apenas `transform`.

In [ ]:
scaler = StandardScaler()

X_cluster_scaled = scaler.fit_transform(X_cluster)

In [ ]:
# Outras opções:
# scaler = MinMaxScaler()
# scaler = RobustScaler()

## 7. Método do cotovelo

O método do cotovelo ajuda a escolher um número inicial de clusters.

A ideia é observar o ponto em que a redução da inertia começa a perder força.

In [ ]:
inertias = []

k_values = range(1, 11)

for k in k_values:
    model_k = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model_k.fit(X_cluster_scaled)
    inertias.append(model_k.inertia_)

plt.plot(k_values, inertias, marker="o")
plt.xlabel("Número de clusters")
plt.ylabel("Inertia")
plt.title("Método do Cotovelo")
plt.show()

## 8. Treinar modelo baseline de clusterização

O KMeans é um algoritmo clássico de clusterização.

Hiperparâmetros:

- `n_clusters`: quantidade de grupos desejada;
- `n_init`: número de inicializações diferentes;
- `random_state`: garante reprodutibilidade.

In [ ]:
cluster_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

cluster_model.fit(X_cluster_scaled)

## 9. Gerar clusters

Após o treinamento, cada registro recebe um número de cluster.

In [ ]:
clusters = cluster_model.predict(X_cluster_scaled)

df_resultado = df.copy()
df_resultado["cluster"] = clusters

df_resultado.head()

## 10. Avaliação da clusterização

Em clusterização, normalmente não existe classe correta.

Portanto, não usamos acurácia, precisão, recall ou matriz de confusão.

Métricas úteis:

- **Silhouette Score**: qualidade da separação dos grupos;
- **Inertia**: soma das distâncias dos pontos aos centroides;
- **Análise visual**;
- **Perfil médio dos clusters**.

In [ ]:
silhouette = silhouette_score(X_cluster_scaled, clusters)

print("Silhouette Score:", silhouette)
print("Inertia:", cluster_model.inertia_)

Interpretação do Silhouette Score:

- próximo de 1: clusters bem separados;
- próximo de 0: clusters sobrepostos;
- próximo de -1: agrupamento ruim.

## 11. Visualizar clusters

Se houver muitas variáveis, podemos usar PCA para reduzir para duas dimensões apenas para visualização.

In [ ]:
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_cluster_scaled)

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=clusters
)

plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.title("Visualização dos Clusters com PCA")
plt.show()

## 12. Analisar perfil dos clusters

A interpretação dos clusters é uma etapa essencial.

O número do cluster não tem significado sozinho.

Precisamos analisar as médias, medianas e características de cada grupo.

In [ ]:
perfil_clusters = df_resultado.groupby("cluster").mean(numeric_only=True)

perfil_clusters

In [ ]:
quantidade_por_cluster = df_resultado["cluster"].value_counts().sort_index()

quantidade_por_cluster

## 13. Predição de cluster para novo registro

Criamos um registro simples usando a mediana das variáveis.

Importante: o novo registro precisa passar pela mesma normalização usada no treinamento.

In [ ]:
novo_registro = pd.DataFrame([{
    col: X_cluster[col].median() if np.issubdtype(X_cluster[col].dtype, np.number) else X_cluster[col].mode()[0]
    for col in X_cluster.columns
}])

novo_registro_scaled = scaler.transform(novo_registro)

cluster_previsto = cluster_model.predict(novo_registro_scaled)

print("Novo registro:")
print(novo_registro)

print("\nCluster previsto:", cluster_previsto[0])

## Sugestões didáticas

1. Explique que clusterização não possui variável-alvo.
2. Evite falar em percentual de acerto.
3. Use normalização obrigatoriamente com KMeans.
4. Use o método do cotovelo para discutir `n_clusters`.
5. Use o Silhouette Score para discutir qualidade do agrupamento.
6. Mostre que o significado dos clusters vem da análise posterior dos grupos.